# S20 – Assay unification and verification

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from S20_comments import Audit, apply_comments   # the code of the comments of the step, apart from the data handling

INPUT = "../../S19_Biological_annotation/S19.xlsx"
OUTPUT = "../S20.xlsx"
COMMENTS = "../S20_comments.xlsx"
AUDIT = "../S20_audit.xlsx"
STEP = "S20"

df = pd.read_excel(INPUT, keep_default_na=False, na_values=[""]).astype(object)
print(len(df), "records before the step")
audit = Audit(STEP, df, before=INPUT)
put, drop = audit.put, audit.drop                  # a rule sets a value / removes records through the audit, which records the change
changes, removed = audit.changes, audit.removed    # the ledger of the step

3001 records before the step


### 1. Values from the publication review: the comments of this step

In [2]:
# every value from the publication review: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S20_curated_LT_test_raw', 'S20_curated_L1_doi', 'S20_curated_L6_test', 'S20_curated_L7_row'])

11490 comments applied (S20_curated_LT_test_raw, S20_curated_L1_doi, S20_curated_L6_test, S20_curated_L7_row): 11490 values set, 0 records removed


### 2. Corrections from the check against the publications: the comments of this step

In [3]:
# every correction from the check of this field against the publications: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S20_publication_check_01', 'S20_publication_check_02'])

506 comments applied (S20_publication_check_01, S20_publication_check_02): 506 values set, 0 records removed


### 3. Assay readout: one spelling for all sources

In [4]:
READOUT_SPELLING = {   # readout in the source -> (readout, basis)
    'tetrazolium salt': ('TetrazoliumSalt',
                          'the same readout as the Gul et al. label; spelling of the Labouta et al. source'),
    'Alamar blue': ('AlamarBlue',
                     'the same readout as the Gul et al. label; spelling of the Labouta et al. source'),
    'luciferase': ('LuciferaseEnzyme',
                    'the same readout as the Gul et al. label (ATP-dependent luciferase luminescence); spelling of the Labouta et al. '
                    'source'),
    'calcein AM': ('CalceinAM',
                    'the same readout as the Gul et al. label; spelling of the Labouta et al. source'),
    'toluylene red': ('NeutralRed',
                       'toluylene red is the chemical name of the neutral red dye (3-amino-7-dimethylamino-2-methylphenazine hydrochloride)'),
    'LDH activity assay kit': ('LDHrelease',
                                "both Labouta et al. publications measure LDH released into the medium: 10.1016/j.tiv.2009.06.001 p. 2 'LDH leakage "
                                "from damaged cells into the surrounding culture medium'; 10.1021/tx400231u p. 18 'LDH release was measured in "
                                "supernatants'"),
}
for i in df.index:
    v = df.at[i, "Test_Indicator"]
    if not pd.isna(v) and str(v) in READOUT_SPELLING:
        to, basis = READOUT_SPELLING[str(v)]
        put(i, "Test_Indicator", to, f"{STEP}_indicator_vocabulary", "one spelling of the assay readout for all sources", basis, outside=False)

### The table after the step and its audit

In [5]:
# the table after the step; the audit module compares it with the table before the step and writes the audit workbook
df.to_excel(OUTPUT, index=False)
audit.write(AUDIT)

0 records removed and 11996 cells changed compared with the table before the step


3001 records after the step | 0 records removed | 12377 cells changed | ../S20_audit.xlsx
